<a href="https://colab.research.google.com/github/lseidy/BI-Data-Sience/blob/main/whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4 — Whisper na prática

Vocês vão rodar o **Whisper** (OpenAI) em 5 áudios de clientes do RápidoJá pedindo, cancelando ou reclamando. Cada áudio tem uma característica diferente — pra vocês sentirem onde o modelo brilha e onde ele falha.

**O que vocês vão fazer:**

1. Setup (instala biblioteca, baixa modelo `base`).
2. Carregar os 5 áudios.
3. Rodar a transcrição padrão (áudio 1, modelo `base`).
4. **🟧 Mod 1** — trocar o áudio. Ver como cada um se sai.
5. **🟧 Mod 2** — trocar o tamanho do modelo (`tiny`, `base`, `small`). Comparar qualidade × velocidade.
6. **🟧 Mod 3 (opcional)** — forçar `idioma="pt"` em áudios curtos onde o auto-detect falha.

Quem termina rápido: abre o **🌿 Aprofundamento** no final.

## Setup

Roda essas 3 células uma vez. A primeira instala (~30s), a segunda importa, a terceira baixa o modelo `base` (~140MB, ~30s).

In [ ]:
# Instala o openai-whisper.
# A biblioteca traz o modelo, o pipeline de áudio e tudo mais que precisamos.
!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.6 MB/s eta 0:00:00


In [ ]:
# Imports usados no notebook inteiro.
import whisper
import time
import os
from IPython.display import Audio, display

In [ ]:
# Carrega o modelo Whisper.
# 'base' = 74M parâmetros, ~140MB. Bom equilíbrio velocidade × qualidade pra Colab CPU.
# Primeira execução baixa do servidor da OpenAI (~30s); depois fica em cache.

TAMANHO = "base"
modelo = whisper.load_model(TAMANHO)

print(f"✅ Modelo Whisper-{TAMANHO} carregado.")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 100MiB/s]


✅ Modelo Whisper-base carregado.


## Carregar os áudios

5 áudios de clientes do RápidoJá:

| # | Arquivo | Característica |
|---|---|---|
| 1 | `pratica_01_pizza.mp3` | Pedido simples, fala clara |
| 2 | `pratica_02_acai.mp3` | Multi-itens (açaí + complementos) |
| 3 | `pratica_03_cancela.mp3` | Cancelamento (não é pedido!) |
| 4 | `pratica_04_reclamacao.mp3` | Reclamação sobre atraso |
| 5 | `pratica_05_gaucho.mp3` | Regionalismo gaúcho ("bah", "tu") |

A célula abaixo clona o repositório dos áudios direto pra pasta `audios/`. Roda uma vez.


In [ ]:
# Clona os áudios direto do repo público (--depth 1 = só último commit, rápido).

import os

if not os.path.exists("audios"):
    !git clone --depth 1 https://github.com/atbender/pln-au04-audios.git audios
else:
    print("✓ Pasta 'audios' já existe — pulando clone.")

print("\n📁 Áudios disponíveis:")
for f in sorted(os.listdir("audios")):
    if f.endswith(".mp3"):
        print(f"  - {f}")


✓ Pasta 'audios' já existe — pulando clone.

📁 Áudios disponíveis:
  - aps_01_simples.mp3
  - aps_02_multi.mp3
  - aps_03_restaurante.mp3
  - aps_04_endereco_impreciso.mp3
  - aps_05_ambiguo.mp3
  - pratica_01_pizza.mp3
  - pratica_02_acai.mp3
  - pratica_03_cancela.mp3
  - pratica_04_reclamacao.mp3
  - pratica_05_gaucho.mp3


### Ouvir o áudio 1

Antes de transcrever, ouçam pra ter o que comparar.

In [ ]:
display(Audio("audios/pratica_05_gaucho.mp3"))

## Função `transcrever()`

Função pronta. Recebe o caminho do áudio e o modelo Whisper, retorna a transcrição. Imprime também o idioma detectado e quanto tempo levou.

In [ ]:
# Função que transcreve um áudio. Totalmente implementada.
# - arquivo: caminho do .mp3
# - modelo: o modelo Whisper já carregado
# - idioma: 'pt', 'en', None (auto-detect). Default = None.

def transcrever(arquivo, modelo, idioma=None):
    inicio = time.time()

    # Chama o Whisper. Se idioma=None, ele detecta sozinho.
    resultado = modelo.transcribe(arquivo, language=idioma)

    duracao = time.time() - inicio

    print(f"📁 Arquivo: {arquivo}")
    print(f"🌐 Idioma detectado: {resultado['language']}")
    print(f"⏱  Tempo de transcrição: {duracao:.1f}s")
    print(f"\n📝 Transcrição:")
    print(f"   {resultado['text'].strip()}")

    return resultado['text'].strip()

## Transcrição padrão

Áudio 1 (pedido simples) com modelo `base` e auto-detect de idioma.

In [ ]:
ARQUIVO = "audios/pratica_01_pizza.mp3"

texto = transcrever(ARQUIVO, modelo)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/pratica_01_pizza.mp3
🌐 Idioma detectado: pt
⏱  Tempo de transcrição: 7.7s

📝 Transcrição:
   Boa noite! Queria pedir uma pizza grande de calabresa para a rua das flores 234? Apartamento 500 e 2.


### 🟧 Mod 1 — Trocar de áudio

Por padrão, o áudio **5** (gaúcho) está descomentado. Comente esse e descomente outro pra testar.

**Pergunta plantada:** *o áudio 5 tem "bah", "tu", "botar". O Whisper transcreveu certo?*

In [ ]:
# 🟧 MOD 1 — descomente o áudio que quer testar (e comente os outros)

# ARQUIVO = "audios/pratica_01_pizza.mp3"      # pedido simples
# ARQUIVO = "audios/pratica_02_acai.mp3"       # multi-itens
ARQUIVO = "audios/pratica_03_cancela.mp3"    # cancelamento
# ARQUIVO = "audios/pratica_04_reclamacao.mp3" # reclamação de atraso
# ARQUIVO = "audios/pratica_05_gaucho.mp3"     # regionalismo gaúcho

# Ouve antes
display(Audio(ARQUIVO))

# Depois transcreve
texto = transcrever(ARQUIVO, modelo)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/pratica_03_cancela.mp3
🌐 Idioma detectado: pt
⏱  Tempo de transcrição: 9.2s

📝 Transcrição:
   Cancela meu pedido por favor? Esqueci que tenho jantar fora hoje.


### 🟧 Mod 2 — Trocar o tamanho do modelo

Vamos comparar `tiny` (rápido, qualidade aceitável) com `small` (mais lento, qualidade muito boa). O `base` que vocês já têm fica como referência.

**O que esperar:** o `tiny` pode errar em áudios mais difíceis (regionalismo, multi-itens). O `small` melhora bastante mas demora mais. **Pergunta plantada:** *em qual áudio a diferença foi maior?*

In [ ]:
# Carrega os outros tamanhos pra comparação.
# tiny = ~75MB, small = ~480MB. Primeira execução baixa.

modelo_tiny = whisper.load_model("tiny")
modelo_small = whisper.load_model("small")

print("✅ Modelos tiny e small carregados.")

100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 164MiB/s]
100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 70.3MiB/s]


✅ Modelos tiny e small carregados.


In [ ]:
# 🟧 MOD 2 — escolhe o áudio e roda os 3 modelos lado a lado.
# Comparem as 3 transcrições. Quem acertou mais? Quem foi mais rápido?

ARQUIVO = "audios/pratica_05_gaucho.mp3"  # 🟧 troque pra outro áudio também

display(Audio(ARQUIVO))

print("=== Whisper TINY (rápido, menor) ===\n")
transcrever(ARQUIVO, modelo_tiny)

print("\n\n=== Whisper BASE (referência) ===\n")
transcrever(ARQUIVO, modelo)

print("\n\n=== Whisper SMALL (lento, mais preciso) ===\n")
transcrever(ARQUIVO, modelo_small)

=== Whisper TINY (rápido, menor) ===



/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/pratica_05_gaucho.mp3
🌐 Idioma detectado: pt
⏱  Tempo de transcrição: 2.4s

📝 Transcrição:
   Bá? Mandar uma quentinha do executivo lá da Tia Niuda. Tu pode botar bastante a Rose?


=== Whisper BASE (referência) ===



/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/pratica_05_gaucho.mp3
🌐 Idioma detectado: pt
⏱  Tempo de transcrição: 6.1s

📝 Transcrição:
   Bá, manda uma quentinha do executivo lá da Tia Nilda. Tu pode botar bastante arroz?


=== Whisper SMALL (lento, mais preciso) ===



/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/pratica_05_gaucho.mp3
🌐 Idioma detectado: pt
⏱  Tempo de transcrição: 18.3s

📝 Transcrição:
   Bah? Manda uma quentinha do executivo lá da Tia Nilda. Tu pode botar bastante arroz?


'Bah? Manda uma quentinha do executivo lá da Tia Nilda. Tu pode botar bastante arroz?'

### 🟧 Mod 3 (opcional) — Forçar idioma

O auto-detect do Whisper olha os primeiros segundos do áudio pra identificar o idioma. Em áudios curtos, ele pode errar — especialmente em frases com muitas palavras emprestadas do inglês.

Forçar `idioma="pt"` resolve. Em produção, **se você sabe o idioma, sempre force**.

In [ ]:
# 🟧 MOD 3 — força idioma="pt" e veja se muda algo.
# Em áudios curtos ou com termos em inglês, isso costuma melhorar.

ARQUIVO = "audios/pratica_03_cancela.mp3"

print("=== Auto-detect ===\n")
transcrever(ARQUIVO, modelo, idioma=None)

print("\n\n=== Forçando português ===\n")
transcrever(ARQUIVO, modelo, idioma="es")

=== Auto-detect ===



/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/pratica_03_cancela.mp3
🌐 Idioma detectado: pt
⏱  Tempo de transcrição: 4.7s

📝 Transcrição:
   Cancela meu pedido por favor? Esqueci que tenho jantar fora hoje.


=== Forçando português ===



/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📁 Arquivo: audios/pratica_03_cancela.mp3
🌐 Idioma detectado: es
⏱  Tempo de transcrição: 3.4s

📝 Transcrição:
   Cancela mi pedido por favor? Esqueci que tengo jantar fuera hoy.


'Cancela mi pedido por favor? Esqueci que tengo jantar fuera hoy.'

### 💡 Pra fechar

Vocês acabaram de:
- Carregar 3 modelos Whisper de tamanhos diferentes.
- Transcrever áudios reais com características variadas (limpo, multi-item, cancelamento, reclamação, regionalismo).
- Comparar qualidade × velocidade entre tamanhos.
- Ver onde o auto-detect ajuda e onde forçar o idioma é melhor.

**Conexão com IoT:** todo speech-to-text de smart speaker, assistente de carro, ou app de voz passa por um pipeline parecido com esse — só muda o tamanho do modelo (edge × cloud) e o pós-processamento (LLM por trás corrigindo).

**Próxima atividade:** APS 2 — vocês vão pegar 5 áudios novos (pedidos por voz) e usar Whisper + LLM pra extrair JSON estruturado.

In [ ]:
# === LLM via API do Google AI Studio ===
#
# Como conseguir uma API key grátis:
# 1. Acessa https://aistudio.google.com/app/apikey
# 2. Login com Google → "Create API key" → "Create API key in new project"
# 3. Copia a chave que aparecer
# 4. No Colab: ícone de chave (🔑) na barra lateral esquerda → "Add new secret"
# 5. Nome: GEMINI_API_KEY, valor: sua key, liga o toggle "Notebook access"

!pip install -q google-generativeai

from google.colab import userdata
import google.generativeai as genai
import json

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

# Gemini 2.5 Flash com saída forçada em JSON (sem isso, às vezes ele mete ```json em volta).
modelo_llm = genai.GenerativeModel(
    "gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"}
)

# Prompt: análise de polaridade com saída estruturada.
PROMPT = """Classifique a polaridade da review abaixo como "positiva", "negativa" ou "neutra".
Devolva JSON nesse formato:
{
"polaridade": "positiva" | "negativa" | "neutra",
"confianca": float entre 0.0 e 1.0,
"justificativa": breve string explicando o porquê
}

Review: "<<REVIEW>>"
"""

# Exemplo com 3 reviews — uma de cada polaridade pra ver o modelo trabalhando.
reviews = [
    "Pizza chegou fria e o atendimento foi péssimo. Nunca mais peço aqui.",
    "Comida divina! Entrega rápida, embalagem caprichada. Voltarei sempre.",
    "Pedido normal, comida ok, chegou no horário previsto."
]

for review in reviews:
    resposta = modelo_llm.generate_content(PROMPT.replace("<<REVIEW>>", review))
    resultado = json.loads(resposta.text)
    print(f"📝 {review}")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))
    print()

📝 Pizza chegou fria e o atendimento foi péssimo. Nunca mais peço aqui.
{
  "polaridade": "negativa",
  "confianca": 0.99,
  "justificativa": "A review contém múltiplos indicadores negativos, como 'pizza chegou fria', 'atendimento foi péssimo' e a declaração final 'Nunca mais peço aqui', que expressam total insatisfação."
}

📝 Comida divina! Entrega rápida, embalagem caprichada. Voltarei sempre.
{
  "polaridade": "positiva",
  "confianca": 0.98,
  "justificativa": "A review utiliza termos extremamente positivos como \"divina\", \"rápida\" e \"caprichada\", além de expressar intenção de retorno (\"Voltarei sempre\"), indicando alta satisfação."
}

📝 Pedido normal, comida ok, chegou no horário previsto.
{
  "polaridade": "neutra",
  "confianca": 0.9,
  "justificativa": "As expressões 'normal' e 'comida ok' indicam uma experiência sem grandes destaques positivos ou negativos, apenas cumprindo o esperado. 'Chegou no horário previsto' reforça a ausência de problemas, mas não denota uma satis

---

## 🌿 Aprofundamento (em casa)

Pra quem quiser ir além — não precisa fazer em sala:

1. **Gravar o próprio áudio.** No Colab, use `from google.colab import files; files.upload()` pra subir um .mp3 ou .wav que você gravou. Teste sotaques, jargões da sua empresa, frases longas.

2. **Adicionar ruído.** Gere um áudio com TV/música ao fundo (ou edite no Audacity adicionando ruído branco) e veja como o Whisper se sai. Compare com o áudio limpo.

3. **Whisper-large-v3.** Mude pro runtime com GPU (`Runtime → Change runtime type → T4 GPU`) e carregue `whisper.load_model("large-v3")`. Compare com `small` em algum áudio difícil.

4. **Comparar com Gemini.** O Gemini API tem speech-to-text nativo. Mande o mesmo áudio pros dois e compare a transcrição. Qual ganha em PT-BR?

5. **Pós-processamento por LLM.** Pegue uma transcrição com erro (regionalismo ou termo técnico) e mande pro Gemini com prompt: *"corrija a transcrição abaixo mantendo o sentido original"*. Esse é o pattern padrão em produção.